# 🚀 Claude Code Agent no Google Colab com Ollama

Este notebook configura o ambiente para executar o Claude Code como um agente de IA no Google Colab, utilizando o **Ollama** de forma direta.

**Objetivo:** Facilitar a alternância entre diferentes modelos (como o Llama3:8b padrão ou outros de sua escolha) configurando as variáveis de ambiente automaticamente via funções Python.

### 🟢 Célula 1 — instalação + servidor

In [ ]:
%%bash
apt-get update -y
apt-get install -y zstd curl

# instalar ollama
curl -fsSL https://ollama.com/install.sh | sh

# iniciar servidor
nohup /usr/local/bin/ollama serve > ollama.log 2>&1 &
sleep 10

### 🔵 Célula 2 — baixar modelo

In [ ]:
! /usr/local/bin/ollama pull llama3:8b

### 🟣 Célula 3 — teste rápido

In [ ]:
! /usr/local/bin/ollama run llama3:8b "Explique IA em uma frase"

## 🟡 4. Clonar e Preparar o Agente (Claude Code)

Vamos clonar o repositório e instalar as dependências do Node.js necessárias para rodar o agente.

In [ ]:
%%bash
# Lógica de Auto-Update: Se a pasta já existir, atualiza. Se não, clona.
if [ -d "claude-code-agent-colab" ]; then 
  echo "--- Repositório já existe. Atualizando para a versão mais recente... ---"
  cd claude-code-agent-colab && git pull origin master
else
  echo "--- Clonando o repositório pela primeira vez... ---"
  git clone https://github.com/dragonbrxos/claude-code-agent-colab.git
fi

# Instalar dependências de forma otimizada para o Colab
cd claude-code-agent-colab
npm install --no-audit --no-fund --quiet

# Gerar o build ignorando erros de tipos (TypeScript)
npm run build

## 🟠 5. Funções de Configuração de Modelo

Aqui definimos as funções para configurar o modelo padrão ou um modelo customizado.

In [ ]:
import os
import subprocess

def configurar_modelo_padrao():
    """Configura o modelo Llama3:8b como padrão."""
    modelo = "llama3:8b"
    print(f"--- Baixando e configurando modelo padrão: {modelo} ---")
    # Usar subprocess para capturar a saída corretamente no Colab
    subprocess.run(["ollama", "pull", modelo])
    os.environ['ANTHROPIC_BASE_URL'] = 'http://localhost:11434/v1'
    os.environ['ANTHROPIC_API_KEY'] = 'ollama'
    os.environ['CLAUDE_MODEL'] = modelo
    print(f"✅ Modelo {modelo} configurado com sucesso!")

def configurar_modelo_customizado(nome_modelo):
    """Configura qualquer outro modelo disponível no Ollama."""
    print(f"--- Baixando e configurando modelo customizado: {nome_modelo} ---")
    subprocess.run(["ollama", "pull", nome_modelo])
    os.environ['ANTHROPIC_BASE_URL'] = 'http://localhost:11434/v1'
    os.environ['ANTHROPIC_API_KEY'] = 'ollama'
    os.environ['CLAUDE_MODEL'] = nome_modelo
    print(f"✅ Modelo {nome_modelo} configurado com sucesso!")

# Escolha qual função rodar:
configurar_modelo_padrao()

# Exemplo para outro modelo (descomente para usar):
# configurar_modelo_customizado("mistral")

## 🚀 6. Executar o Agente

Agora que o modelo está configurado, vamos rodar o Claude Code.

In [ ]:
print("Iniciando o agente...")
!cd claude-code-agent-colab && node dist/main.js

## 💡 Observações Importantes

*   **Recursos do Colab:** Recomenda-se o uso de um runtime com GPU para melhor performance.
*   **Persistência:** O ambiente do Colab é efêmero. Se o runtime for reiniciado, execute os passos novamente.
*   **Sem Bifrost:** Esta versão utiliza a compatibilidade nativa do Ollama com a API da OpenAI/Anthropic.